# Dán nhãn dữ liệu bằng ChatGPT/OpenAI

Notebook này dùng OpenAI Responses API để dán nhãn theo bộ nhãn DS108. Quy tắc được đọc từ codebook trong `guidelines/` và prompt trong `prompts/`, vì vậy không tạo một taxonomy riêng.

Quy trình khuyến nghị: chạy thử một số dòng → kiểm tra kết quả → chạy batch → dùng người gán nhãn kiểm tra mẫu và tính IAA trước khi coi kết quả là gold label. LLM chỉ là công cụ tiền gán nhãn, không thay thế bước xác nhận của con người.

In [ ]:
# Chạy một lần trong Google Colab/Jupyter có Internet
%pip install -q -U openai pandas

In [ ]:
import hashlib
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

# Có thể đổi sang model khác được bật trong tài khoản.
MODEL_NAME = "gpt-5"
PROMPT_VERSION = "ds108-compact-v1"
REQUEST_INTERVAL_SECONDS = 0.0
MAX_ROWS = None  # Ví dụ: 100 để chạy pilot

# Tự tìm thư mục Annotation để notebook chạy được từ root, notebooks/ hoặc Colab.
def find_annotation_dir():
    candidates = [Path.cwd(), Path.cwd() / "Annotation", Path.cwd().parent, Path.cwd().parent / "Annotation", Path("/content/Annotation")]
    for candidate in candidates:
        if (candidate / "guidelines" / "label_definitions.md").exists():
            return candidate.resolve()
    raise FileNotFoundError("Không tìm thấy thư mục Annotation đã tổ chức.")

ANNOTATION_DIR = find_annotation_dir()
INPUT_CSV = ANNOTATION_DIR.parent / "Preprocessing" / "data" / "raw.csv"
OUTPUT_CSV = ANNOTATION_DIR / "outputs" / "openai_annotations.csv"

# Ưu tiên Colab Secret, sau đó mới đọc biến môi trường; không in API key ra màn hình.
def get_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.getenv("OPENAI_API_KEY")

api_key = get_api_key()
if not api_key:
    raise RuntimeError("Chưa đặt OPENAI_API_KEY trong Colab Secret hoặc biến môi trường.")
client = OpenAI(api_key=api_key)
print(f"Model: {MODEL_NAME}")
print(f"Annotation directory: {ANNOTATION_DIR.resolve()}")

In [ ]:
# Đọc các tài liệu chuẩn dùng chung với quy trình Gemini/human annotation.
def read_utf8(path):
    return Path(path).read_text(encoding="utf-8")

LABEL_DEFINITIONS = read_utf8(ANNOTATION_DIR / "guidelines" / "label_definitions.md")
PROMPT_TEMPLATE = read_utf8(ANNOTATION_DIR / "prompts" / "prompt_dan_nhan_llm.md")
CANONICAL_SCHEMA = json.loads(read_utf8(ANNOTATION_DIR / "schemas" / "annotation_schema.json"))

# Structured Outputs cần schema strict với tất cả thuộc tính nằm trong required.
OPENAI_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "toxicity": {"type": "string", "enum": ["TOXIC", "NON-TOXIC"]},
        "expression_type": {"type": ["string", "null"], "enum": ["EXPLICIT", "IMPLICIT", None]},
        "evidence": {"type": "string"},
        "needs_review": {"type": "boolean"},
        "review_reason": {"type": ["string", "null"]},
    },
    "required": ["toxicity", "expression_type", "evidence", "needs_review", "review_reason"],
}

def validate_annotation(item):
    required = set(OPENAI_SCHEMA["required"])
    if set(item) != required:
        raise ValueError(f"Sai field output: {sorted(set(item) ^ required)}")
    toxicity = item["toxicity"]
    expression = item["expression_type"]
    if toxicity == "NON-TOXIC" and expression is not None:
        raise ValueError("NON-TOXIC phải có expression_type=null")
    if toxicity == "TOXIC" and expression not in {"EXPLICIT", "IMPLICIT"}:
        raise ValueError("TOXIC phải có EXPLICIT hoặc IMPLICIT")
    if item["needs_review"] and not item["review_reason"]:
        raise ValueError("needs_review=true phải có review_reason")
    if not item["needs_review"] and item["review_reason"] is not None:
        raise ValueError("needs_review=false phải có review_reason=null")
    return item

def input_hash(comment):
    return hashlib.sha256(str(comment).encode("utf-8")).hexdigest()

print("Đã nạp label definitions, prompt và schema chuẩn.")

In [ ]:
SYSTEM_PROMPT = f"""Bạn là annotator cho dự án DS108. Chỉ sử dụng taxonomy và quy tắc trong tài liệu dưới đây. Không tự thêm nhãn cảm xúc, chủ đề hoặc category. Trả về đúng JSON theo schema Structured Outputs.

=== LABEL DEFINITIONS ===
{LABEL_DEFINITIONS}
=== OPERATIONAL PROMPT ===
{PROMPT_TEMPLATE}
"""

def render_input(comment):
    prompt = PROMPT_TEMPLATE.replace("{comment}", str(comment))
    return (
        prompt
        + "\n\nHãy xử lý bản ghi sau và chỉ trả về JSON theo schema đã cung cấp:\n"
        + json.dumps({"comment": str(comment)}, ensure_ascii=False)
    )

def annotate_one(comment):
    response = client.responses.create(
        model=MODEL_NAME,
        input=[
            {"role": "system", "content": [{"type": "input_text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [{"type": "input_text", "text": render_input(comment)}]},
        ],
        text={"format": {"type": "json_schema", "name": "ds108_annotation", "strict": True, "schema": OPENAI_SCHEMA}},
    )
    if getattr(response, "status", None) == "incomplete":
        raise RuntimeError("OpenAI trả về incomplete response")
    raw = response.output_text
    item = validate_annotation(json.loads(raw))
    return item, raw

# Smoke test: chỉ gọi API khi chạy cell này.
# test_item, test_raw = annotate_one("Nội dung mẫu cần kiểm tra.")
# print(json.dumps(test_item, ensure_ascii=False, indent=2))

In [ ]:
# Nạp dữ liệu đầu vào. CSV cần tối thiểu cột comment; row_id/platform là tùy chọn.
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Không tìm thấy INPUT_CSV: {INPUT_CSV.resolve()}")

df = pd.read_csv(INPUT_CSV)
if "comment" not in df.columns:
    raise ValueError("CSV phải có cột comment")
if "row_id" not in df.columns:
    df.insert(0, "row_id", range(len(df)))
if "platform" not in df.columns:
    df["platform"] = ""

df["comment"] = df["comment"].fillna("").astype(str)
df = df[df["comment"].str.strip().ne("")].copy()
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()
print(f"Số bản ghi sẽ xử lý: {len(df):,}")

In [ ]:
# Chạy batch. Kết quả lỗi được giữ dưới dạng needs_review để không làm mất dòng.
records = []
for index, row in df.iterrows():
    comment = row["comment"]
    base = {
        "row_id": row["row_id"],
        "platform": row["platform"],
        "comment": comment,
        "input_hash": input_hash(comment),
        "model": MODEL_NAME,
        "prompt_version": PROMPT_VERSION,
        "api": "OpenAI Responses API",
    }
    try:
        annotation, raw = annotate_one(comment)
        base.update(annotation)
        base["raw_response"] = raw
    except Exception as exc:
        base.update({
            "toxicity": None,
            "expression_type": None,
            "evidence": "",
            "needs_review": True,
            "review_reason": f"API/parse error: {type(exc).__name__}: {exc}",
            "raw_response": "",
        })
    records.append(base)
    if (len(records) % 25 == 0) or (len(records) == len(df)):
        print(f"Đã xử lý {len(records):,}/{len(df):,}")
    if REQUEST_INTERVAL_SECONDS:
        time.sleep(REQUEST_INTERVAL_SECONDS)

result_df = pd.DataFrame(records)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
result_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {OUTPUT_CSV.resolve()}")
print(result_df["toxicity"].value_counts(dropna=False))

## Kiểm tra sau khi chạy

- Lọc `needs_review == True` để người gán nhãn xử lý thủ công.
- Lấy một mẫu của cả `TOXIC` và `NON-TOXIC` để kiểm tra bằng người.
- Không dùng trực tiếp toàn bộ output làm gold label nếu chưa có người xác nhận, adjudication và báo cáo IAA.
- Dùng `tools/evaluate_iaa.py` để so sánh hai file annotation của người trên các cột nhãn canonical.